# z-expansion parquet sanity check

Minimal check of the MINERvA z-expansion columns in `spline_weights_df.parquet`, directly from `create_rw_syst_df.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.file_locations import intermediate_files_location

parquet_path = Path(f"{intermediate_files_location}/spline_weights_df.parquet")
assert parquet_path.exists(), f"Missing parquet file: {parquet_path}"
parquet_path


In [ ]:
zexp_cols = [
    "weight_minerva_FA",
    "weight_spline_FAzexpPCA1",
    "weight_spline_FAzexpPCA2",
    "weight_spline_FAzexpPCA3",
    "weight_spline_FAzexpPCA4",
]
base_cols = ["filetype", "run", "subrun", "event", "MaCCQE_UBGenie"]
cols = base_cols + zexp_cols

scan = pl.scan_parquet(parquet_path)
schema = scan.collect_schema()
print(f"Rows: {scan.select(pl.len()).collect().item():,}")
print([c for c in schema.names() if "FA" in c or "zexp" in c])
missing = [c for c in cols if c not in schema.names()]
assert not missing, f"Missing expected columns: {missing}"
schema.select(cols) if hasattr(schema, 'select') else {c: schema[c] for c in cols}


In [ ]:
# Keep the notebook lightweight even for full production parquets.
MAX_ROWS = 200_000
df = scan.select(cols).head(MAX_ROWS).collect()
print(f"Loaded {df.height:,} rows for this sanity check")

for row in df.group_by("filetype").len().sort("filetype").iter_rows(named=True):
    print(f"{row['filetype']}: {row['len']:,}")

cv = df["weight_minerva_FA"].to_numpy().astype(float)
ma = np.asarray(df["MaCCQE_UBGenie"].to_list(), dtype=float)
pca = {c: np.asarray(df[c].to_list(), dtype=float) for c in zexp_cols[1:]}

finite_cv = np.isfinite(cv)
ma_span = np.nanmax(ma, axis=1) - np.nanmin(ma, axis=1)
nonunit_ma = np.any(~np.isclose(ma, 1.0), axis=1)
responsive_ma = ma_span > 1e-6
flat_nonunit_ma = nonunit_ma & ~responsive_ma
check_mask = finite_cv & responsive_ma

print(f"Finite CV fraction: {finite_cv.mean():.4f}")
print(f"Rows with non-unit MaCCQE spline: {nonunit_ma.sum():,} / {len(nonunit_ma):,}")
print(f"Rows with flat non-unit MaCCQE spline: {flat_nonunit_ma.sum():,}")
print(f"Rows with MA-responsive MaCCQE spline: {responsive_ma.sum():,}")
print(f"Rows in z-exp interpolation check mask: {check_mask.sum():,}")

if flat_nonunit_ma.sum():
    w = cv[flat_nonunit_ma & finite_cv]
    print(
        "flat non-unit MaCCQE rows have CV summary: "
        f"min={w.min():.5g}, median={np.median(w):.5g}, mean={w.mean():.5g}, max={w.max():.5g}"
    )
    print("First flat non-unit MaCCQE rows:")
    for row in ma[flat_nonunit_ma][:5]:
        print(" ", row)

if check_mask.sum():
    w = cv[check_mask]
    print(
        "weight_minerva_FA on MA-responsive rows: "
        f"min={w.min():.5g}, median={np.median(w):.5g}, mean={w.mean():.5g}, max={w.max():.5g}"
    )
    print("First MA-responsive MaCCQE rows:")
    for row in ma[check_mask][:5]:
        print(" ", row)
else:
    print("No MA-responsive MaCCQE rows in the loaded sample; increase MAX_ROWS or use a different input file.")

for name, mat in pca.items():
    print(
        f"{name}: shape={mat.shape}, finite={np.isfinite(mat).mean():.4f}, "
        f"max |sigma0 - CV|={np.nanmax(np.abs(mat[:, 3] - cv)):.3g}"
    )


In [ ]:
assert finite_cv.all(), "weight_minerva_FA contains non-finite values"
for name, mat in pca.items():
    assert mat.shape == (len(cv), 7), f"{name} has wrong shape: {mat.shape}"
    assert np.isfinite(mat).all(), f"{name} contains non-finite values"
    assert np.allclose(mat[:, 3], cv), f"{name} zero-sigma column does not match CV"

if responsive_ma.any():
    assert not np.allclose(cv[responsive_ma], 1.0), (
        "Rows with MA-responsive MaCCQE splines still have all-unit z-exp CV weights; "
        "check the z-exp interpolation inputs."
    )

print("Basic parquet z-exp checks passed.")


In [ ]:
plot_mask = check_mask if check_mask.sum() else finite_cv
print(f"Plotting {plot_mask.sum():,} rows ({'MA-responsive' if check_mask.sum() else 'all finite CV'})")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].hist(cv[plot_mask], range=(0, 2), bins=20, histtype="step", linewidth=2)
axes[0].set_xlabel("weight_minerva_FA")
axes[0].set_ylabel("Events")
axes[0].grid(alpha=0.25)

sigma_values = np.array([-3, -2, -1, 0, 1, 2, 3])
for name, mat in pca.items():
    mean_by_sigma = mat[plot_mask].mean(axis=0)
    axes[1].plot(sigma_values, mean_by_sigma, marker="o", label=name.replace("weight_spline_", ""))

axes[1].axhline(cv[plot_mask].mean(), color="black", linewidth=1, alpha=0.5)
axes[1].set_xlabel("PCA sigma point")
axes[1].set_ylabel("Mean event weight")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)

fig.tight_layout()